# TripGraph — Run on Google Colab

This notebook installs dependencies, mounts your Google Drive, sets API credentials,
and launches the Streamlit app with a public URL via **localtunnel**.

**Before running:**
1. Upload the entire `TripGraph/` project folder to `MyDrive/TripGraph/`.
2. Add your secrets in *Colab → Runtime → Secrets (🔑)*:
   - `NEO4J_URI`
   - `NEO4J_PASSWORD`
   - `GROQ_API_KEY`
   - `ORS_API_KEY` *(optional)*
3. Run all cells in order.

## 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/TripGraph'
APP_DIR      = os.path.join(PROJECT_ROOT, 'app')
assert os.path.isdir(APP_DIR), f"App directory not found: {APP_DIR}"
print('Drive mounted. Project root:', PROJECT_ROOT)

## 2 — Install dependencies

In [ ]:
%%capture install_log
!pip install -q \
    streamlit==1.35.0 \
    streamlit-folium==0.20.0 \
    folium==0.16.0 \
    neo4j==5.19.0 \
    groq==0.8.0 \
    scikit-learn==1.4.2 \
    python-dotenv==1.0.1 \
    requests==2.31.0

# localtunnel — zero-config public URL for port 8501
!npm install -q -g localtunnel

print('Installation complete.')

## 3 — Load API credentials from Colab Secrets

In [ ]:
from google.colab import userdata

def _secret(key, default=''):
    try:
        return userdata.get(key)
    except Exception:
        return default

os.environ['NEO4J_URI']      = _secret('NEO4J_URI',      'neo4j+s://XXXXXXXX.databases.neo4j.io')
os.environ['NEO4J_PASSWORD'] = _secret('NEO4J_PASSWORD', '')
os.environ['GROQ_API_KEY']   = _secret('GROQ_API_KEY',   '')
os.environ['ORS_API_KEY']    = _secret('ORS_API_KEY',    '')

print('Credentials loaded from Colab Secrets.')
print('Neo4j URI:', os.environ['NEO4J_URI'])

## 4 — Write a minimal `.env` file for the app

The Streamlit app reads credentials from environment variables (already set above),
but we also write a `.env` so `python-dotenv` finds them when the app boots.

In [ ]:
env_content = f"""NEO4J_URI={os.environ['NEO4J_URI']}
NEO4J_USER=neo4j
NEO4J_PASSWORD={os.environ['NEO4J_PASSWORD']}
GROQ_API_KEY={os.environ['GROQ_API_KEY']}
ORS_API_KEY={os.environ['ORS_API_KEY']}
"""

env_path = os.path.join(APP_DIR, '.env')
with open(env_path, 'w') as f:
    f.write(env_content)
print('Wrote', env_path)

## 5 — Get the Colab public IP (needed for localtunnel password)

In [ ]:
import urllib.request
public_ip = urllib.request.urlopen('https://api.ipify.org').read().decode('utf8')
print('Colab public IP (localtunnel password):', public_ip)

## 6 — Launch Streamlit in the background

In [ ]:
import subprocess, time

streamlit_log = open('/tmp/streamlit.log', 'w')

proc = subprocess.Popen(
    [
        'streamlit', 'run', os.path.join(APP_DIR, 'main.py'),
        '--server.port', '8501',
        '--server.headless', 'true',
        '--server.enableCORS', 'false',
        '--server.enableXsrfProtection', 'false',
    ],
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
    cwd=APP_DIR,
)

time.sleep(6)   # wait for Streamlit to initialise
print('Streamlit PID:', proc.pid)
print('Check /tmp/streamlit.log if you hit errors.')

## 7 — Open a public tunnel via localtunnel

In [ ]:
tunnel_log = open('/tmp/tunnel.log', 'w')

tunnel_proc = subprocess.Popen(
    ['lt', '--port', '8501'],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

time.sleep(4)

with open('/tmp/tunnel.log') as f:
    tunnel_output = f.read()

print(tunnel_output)
print('---')
print(f'When prompted for a password/IP, enter: {public_ip}')

## 8 — (Optional) View Streamlit logs

In [ ]:
with open('/tmp/streamlit.log') as f:
    print(f.read()[-3000:])   # last 3 KB of log

## 9 — Stop the app (run when finished)

In [ ]:
proc.terminate()
tunnel_proc.terminate()
print('Streamlit and tunnel stopped.')